# Severity-Holdout Generalization Control

Trains the augmented arm on mild and moderate corruption only, then tests on severe. Answers whether augmentation transfers to degradation worse than it trained for. Clean-trained comparison rows already exist in the master CSV.


In [1]:
import os
import sys
import shutil
import random

# ---------------------------------------------------------------------------
# Fix: Disable rich/tqdm progress bars that cause RecursionError on Kaggle.
# rich's console proxy enters an infinite loop on Jupyter output streams.
# Reference: https://github.com/openvinotoolkit/anomalib/issues
# ---------------------------------------------------------------------------
os.environ["ANOMALIB_USE_RICH"] = "0"
os.environ["RICH_NO_THEME"] = "1"
sys.setrecursionlimit(5000)  # Guard against any residual recursion

import time
import json
import gc
import numpy as np
import pandas as pd
import cv2
import albumentations as A
import torch
from torch.utils.data import Dataset
from PIL import Image

# ---------------------------------------------------------------------------
# 0. Global Setup & Timeout Logic
# ---------------------------------------------------------------------------
START_TIME = time.time()
TIMEOUT_SECONDS = 11.5 * 3600  # 11.5 hours
OUTPUT_FILE = "results/patchcore_augmented.csv"
PARTIAL_FILE = "results/patchcore_augmented_partial.csv"
AUG_TRAIN_ROOT = "/kaggle/tmp/aug_train"

print(f"Script started at {time.ctime(START_TIME)}")
print(f"Graceful timeout set to {TIMEOUT_SECONDS / 3600:.1f} hours.")

def check_timeout():
    elapsed = time.time() - START_TIME
    if elapsed > TIMEOUT_SECONDS:
        print(f"\n" + "!"*60)
        print(f"TIMEOUT REACHED ({elapsed/3600:.1f}h). Exiting gracefully.")
        print("!"*60)
        save_results()
        sys.exit(0)

def save_results():
    if 'all_results' in globals() and all_results:
        os.makedirs("results", exist_ok=True)
        df = pd.DataFrame(all_results)
        df.to_csv(OUTPUT_FILE, index=False)
        df.to_csv(PARTIAL_FILE, index=False)
        # print(f"✅ Results persisted to {OUTPUT_FILE}")

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---------------------------------------------------------------------------
# 1. Dependency Check
# ---------------------------------------------------------------------------
def ensure_dependencies():
    import subprocess
    packages = ["anomalib", "lightning", "albumentationsx", "scikit-image", "opencv-python-headless"]
    for package in packages:
        try:
            check_name = "cv2" if package == "opencv-python-headless" else (package.replace("-", "_") if package != "albumentationsx" else "albumentations")
            __import__(check_name)
        except ImportError:
            print(f"Installing missing dependency: {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

ensure_dependencies()

# Now safe to import
import lightning as L
from anomalib.data import MVTecAD
from anomalib.engine import Engine
from anomalib.models import Patchcore

# ---------------------------------------------------------------------------
# 2. Embedded Corruption & Rescue Functions
# ---------------------------------------------------------------------------

def apply_low_light(image, gamma, seed=42):
    rng = np.random.default_rng(seed)
    img_float = image.astype(np.float64) / 255.0
    img_dark = np.power(img_float, 1.0 / gamma)
    photon_scale = 50.0
    img_noisy = rng.poisson(np.clip(img_dark * photon_scale, 0, None)) / photon_scale
    return np.clip(img_noisy * 255, 0, 255).astype(np.uint8)

def apply_gaussian_blur(image, sigma, kernel_size):
    return cv2.GaussianBlur(image, (kernel_size, kernel_size), sigmaX=sigma)

def apply_motion_blur(image, kernel_size):
    kernel = np.zeros((kernel_size, kernel_size))
    kernel[int((kernel_size-1)/2), :] = np.ones(kernel_size)
    kernel /= kernel_size
    return cv2.filter2D(image, -1, kernel)

def apply_sensor_noise(image, gauss_var, seed=42):
    """Apply Gaussian noise + 5% salt-and-pepper impulse noise.
    
    Note: The salt-and-pepper component (sp_ratio=0.05) adds impulse noise
    on top of the Gaussian noise. This means NLM denoising is suboptimal
    for this corruption type; a median filter would be more appropriate
    for the impulse component.
    """
    rng = np.random.default_rng(seed)
    img_float = image.astype(np.float64) / 255.0
    noise = rng.normal(0, np.sqrt(gauss_var), img_float.shape)
    img_noisy = img_float + noise
    sp_ratio = 0.05
    salt = rng.random(img_float.shape[:2]) < (sp_ratio / 2)
    pepper = rng.random(img_float.shape[:2]) < (sp_ratio / 2)
    img_noisy[salt] = 1.0
    img_noisy[pepper] = 0.0
    return np.clip(img_noisy * 255, 0, 255).astype(np.uint8)

def apply_fog(image, fog_coef_lower, fog_coef_upper, alpha_coef=0.1, seed=42):
    """Apply synthetic fog/haze using Albumentations RandomFog with deterministic seed."""
    import random as _random
    _random.seed(seed)
    np.random.seed(seed % (2**31))
    t = A.RandomFog(fog_coef_range=(fog_coef_lower, fog_coef_upper),
                    alpha_coef=alpha_coef, p=1.0)
    return t(image=image)["image"]

def apply_corruption(image, ctype, severity, config, seed=42):
    params = config["corruptions"][ctype][severity]
    if ctype == "low_light": return apply_low_light(image, params["gamma"], seed)
    elif ctype == "gaussian_blur": return apply_gaussian_blur(image, params["sigma"], params["kernel_size"])
    elif ctype == "motion_blur": return apply_motion_blur(image, params["kernel_size"])
    elif ctype == "sensor_noise": return apply_sensor_noise(image, params["gauss_var"], seed)
    elif ctype == "fog_haze": return apply_fog(image, params["fog_coef_lower"], params["fog_coef_upper"], params.get("alpha_coef", 0.1), seed=seed)
    else: raise ValueError(f"Unknown corruption type: {ctype}")

def apply_clahe(image: np.ndarray, clip_limit: float = 3.0, tile_grid_size: tuple = (8, 8)) -> np.ndarray:
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    cl = clahe.apply(l)
    lab_enhanced = cv2.merge((cl, a, b))
    bgr_out = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)
    return cv2.cvtColor(bgr_out, cv2.COLOR_BGR2RGB)

def apply_wiener_deconv(image: np.ndarray, sigma: float, kernel_size: int, balance: float = 0.1) -> np.ndarray:
    from skimage.restoration import wiener
    from skimage import img_as_float, img_as_ubyte
    img_float = img_as_float(image)
    ax = np.arange(-kernel_size // 2 + 1, kernel_size // 2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    psf = np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    psf /= psf.sum()
    result = np.zeros_like(img_float)
    for c in range(3):
        result[:, :, c] = wiener(img_float[:, :, c], psf, balance)
    return img_as_ubyte(np.clip(result, 0, 1))

def apply_motion_wiener_deconv(image: np.ndarray, kernel_size: int, balance: float = 0.1) -> np.ndarray:
    from skimage.restoration import wiener
    from skimage import img_as_float, img_as_ubyte
    img_float = img_as_float(image)
    psf = np.zeros((kernel_size, kernel_size))
    psf[kernel_size // 2, :] = 1.0 / kernel_size
    result = np.zeros_like(img_float)
    for c in range(3):
        result[:, :, c] = wiener(img_float[:, :, c], psf, balance)
    return img_as_ubyte(np.clip(result, 0, 1))

def apply_nlm_denoise(image: np.ndarray, patch_size: int = 7, patch_distance: int = 11) -> np.ndarray:
    from skimage.restoration import denoise_nl_means, estimate_sigma
    img_float = image.astype(np.float64) / 255.0
    sigma_est = np.mean(estimate_sigma(img_float, channel_axis=-1))
    denoised = denoise_nl_means(img_float, h=1.15 * sigma_est, patch_size=patch_size, patch_distance=patch_distance, fast_mode=True, channel_axis=-1)
    return np.clip(denoised * 255, 0, 255).astype(np.uint8)

def apply_retinex(image: np.ndarray) -> np.ndarray:
    img_float = image.astype(np.float64) + 1.0
    blurred = cv2.GaussianBlur(img_float, (0, 0), 30)
    log_retinex = np.log10(img_float) - np.log10(blurred + 1.0)
    for i in range(3):
        log_retinex[:,:,i] = (log_retinex[:,:,i] - np.min(log_retinex[:,:,i])) / (np.max(log_retinex[:,:,i]) - np.min(log_retinex[:,:,i])) * 255
    return log_retinex.astype(np.uint8)

def apply_dark_channel_prior_dehaze(image: np.ndarray, omega: float = 0.95, patch_size: int = 15) -> np.ndarray:
    img = image.astype(np.float64) / 255.0
    dark = np.min(img, axis=2)
    from scipy.ndimage import minimum_filter
    dark_channel = minimum_filter(dark, size=patch_size)
    flat_dark = dark_channel.ravel()
    top_indices = np.argsort(flat_dark)[-max(1, int(0.001 * len(flat_dark))):]
    A = np.max(img.reshape(-1, 3)[top_indices], axis=0)
    normed = img / (A + 1e-6)
    normed_dark = np.min(normed, axis=2)
    normed_dark_filtered = minimum_filter(normed_dark, size=patch_size)
    transmission = np.clip(1 - omega * normed_dark_filtered, 0.1, 1.0)
    result = (img - A) / transmission[:, :, np.newaxis] + A
    return np.clip(result * 255, 0, 255).astype(np.uint8)

def get_rescue_map(severity, config):
    """Return rescue methods with PSF parameters matched to the current corruption severity.
    
    This ensures Wiener deconvolution uses the exact PSF that generated the blur,
    rather than a hardcoded severe-tier kernel.
    """
    gauss_p = config["corruptions"]["gaussian_blur"][severity]
    motion_p = config["corruptions"]["motion_blur"][severity]
    return {
        "low_light": [("CLAHE", apply_clahe), ("Retinex", apply_retinex)],
        "gaussian_blur": [("Wiener", lambda img, s=gauss_p["sigma"], k=gauss_p["kernel_size"]:
                           apply_wiener_deconv(img, sigma=s, kernel_size=k))],
        "motion_blur": [("Wiener (Motion PSF)", lambda img, k=motion_p["kernel_size"]:
                         apply_motion_wiener_deconv(img, kernel_size=k))],
        "sensor_noise": [("NLM Denoise", apply_nlm_denoise)],
        "fog_haze": [("Dehaze (Dark Channel)", apply_dark_channel_prior_dehaze)]
    }

# ---------------------------------------------------------------------------
# 3. Load configuration
# ---------------------------------------------------------------------------
CONFIG_PATH = "/kaggle/input/datasets/hasanmahmudabdullah/new-exp-config/experiment_config.json"
with open(CONFIG_PATH) as f: config = json.load(f)

MVTEC_ROOT = config["mvtec_root"]
CATEGORIES = config["categories"]
SEEDS = config["seeds"]

# ---------------------------------------------------------------------------
# 4. Corrupted Dataset Wrapper
# ---------------------------------------------------------------------------
class CorruptedDatasetWrapper(Dataset):
    def __init__(self, base_dataset, ctype, severity, config, rescue_func=None):
        self.base_dataset = base_dataset
        self.ctype = ctype
        self.severity = severity
        self.config = config
        self.rescue_func = rescue_func
    def __len__(self): return len(self.base_dataset)
    def __getattr__(self, name):
        # Transparently proxy any attribute Anomalib expects (collate_fn, transform, etc.)
        # to the underlying base dataset. This prevents AttributeError on Anomalib internals.
        return getattr(self.base_dataset, name)
    def __getitem__(self, idx):
        import dataclasses
        item = self.base_dataset[idx]

        # Anomalib v1.x returns an ImageItem dataclass, not a dict.
        # Use attribute access and dataclasses.replace() to stay compatible.
        if dataclasses.is_dataclass(item):
            image = item.image
        else:
            image = item["image"]

        if isinstance(image, torch.Tensor):
            img_np = image.permute(1, 2, 0).cpu().numpy()
            if img_np.max() <= 1.0: img_np = (img_np * 255).astype(np.uint8)
            else: img_np = img_np.astype(np.uint8)
        else:
            img_np = np.array(image).astype(np.uint8)

        corrupted = apply_corruption(img_np, self.ctype, self.severity, self.config, seed=42+idx)
        final_img = self.rescue_func(corrupted) if self.rescue_func else corrupted
        final_tensor = torch.from_numpy(final_img).permute(2, 0, 1).float() / 255.0

        if dataclasses.is_dataclass(item):
            return dataclasses.replace(item, image=final_tensor)
        else:
            item["image"] = final_tensor
            return item

# ---------------------------------------------------------------------------
class DisableCheckpointing(L.Callback):
    """Strips any ModelCheckpoint callbacks before training starts.
    This avoids the Lightning contradiction where Anomalib adds ModelCheckpoint
    but the trainer also has enable_checkpointing=False."""
    def setup(self, trainer, pl_module, stage):
        from lightning.pytorch.callbacks import ModelCheckpoint
        trainer.callbacks = [
            cb for cb in trainer.callbacks
            if not isinstance(cb, ModelCheckpoint)
        ]

def make_engine():
    """Engine configured to avoid:
    1. RecursionError from rich/tqdm on Kaggle (enable_progress_bar=False)
    2. ModelCheckpoint contradiction error (DisableCheckpointing callback)
    3. Heatmap spam (default_root_dir=/tmp)
    """
    return Engine(
        max_epochs=1,
        accelerator="auto",
        devices=1,
        default_root_dir="/tmp/anomalib",
        enable_progress_bar=False,
        callbacks=[DisableCheckpointing()]
    )

def safe_auroc(result_dict):
    """Extract AUROC from Anomalib result dict. Logs a warning if key not found."""
    for key in ["image_AUROC", "image_auroc", "auroc", "AUROC", "test_image_AUROC"]:
        if key in result_dict:
            return result_dict[key]
    print(f"  ⚠️  WARNING: AUROC key not found in results. Available keys: {list(result_dict.keys())}")
    return None  # None instead of 0 so it's visible in CSV
ROWS_PER_UNIT = 10

# ---------------------------------------------------------------------------
# Generalization control -- experiment configuration
# ---------------------------------------------------------------------------
from anomalib.models import Patchcore, Padim

CATEGORIES = ['carpet', 'grid', 'leather', 'bottle', 'cable', 'capsule', 'hazelnut', 'screw']
SEED = 42
MODELS = ["PatchCore", "PaDiM"]
ALL_CTYPES = ['low_light', 'gaussian_blur', 'motion_blur', 'sensor_noise', 'fog_haze']
ALL_SEVS = ['mild', 'moderate', 'severe']

OUTPUT_FILE = "results/severity_holdout.csv"
PARTIAL_FILE = "results/severity_holdout_partial.csv"
AUG_ROOT_BASE = "/kaggle/tmp/aug_severity_holdout"

os.makedirs("results", exist_ok=True)


def make_model(name):
    """Identical hyperparameters to the published benchmark arms."""
    if name == "PatchCore":
        return Patchcore(backbone="wide_resnet50_2", num_neighbors=9)
    return Padim(backbone="wide_resnet50_2",
                 layers=["layer1", "layer2", "layer3"], n_features=100)


def prepare_augmented_category(category, dst_root, aug_ctypes, aug_sevs,
                               aug_prob=0.5, rng_seed=0):
    """Write an augmented copy of one category, drawing only from aug_ctypes/aug_sevs.

    Deliberately identical to notebook 02's prepare_augmented_train_data() except
    for the restricted draw pools and the per-category destination -- including
    the fixed seed=42 passed to apply_corruption, so this arm carries the same
    known noise-realization defect as the published arm and stays comparable.
    """
    random.seed(rng_seed)
    src_train = os.path.join(MVTEC_ROOT, category, "train", "good")
    dst_train = os.path.join(dst_root, category, "train", "good")
    os.makedirs(dst_train, exist_ok=True)
    n_aug = 0
    for fname in sorted(os.listdir(src_train)):
        if not fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
            continue
        dst_path = os.path.join(dst_train, fname)
        if os.path.exists(dst_path):
            continue
        img_bgr = cv2.imread(os.path.join(src_train, fname))
        if img_bgr is None:
            continue
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        if random.random() < aug_prob:
            ctype = random.choice(aug_ctypes)
            sev = random.choice(aug_sevs)
            img = apply_corruption(img, ctype, sev, config, seed=42)
            n_aug += 1
        cv2.imwrite(dst_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    for subdir in ["test", "ground_truth"]:
        src = os.path.join(MVTEC_ROOT, category, subdir)
        dst = os.path.join(dst_root, category, subdir)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
    print(f"    augmented {n_aug} training images from {aug_ctypes} x {aug_sevs}")


def evaluate(engine, model, base_test_data, ctype, sev):
    """One degradation inference pass on a held-out (corruption, severity)."""
    from torch.utils.data import DataLoader
    loader = DataLoader(
        CorruptedDatasetWrapper(base_test_data, ctype, sev, config),
        batch_size=32, num_workers=4, collate_fn=base_test_data.collate_fn)
    return safe_auroc(engine.test(model=model, dataloaders=loader)[0])


# ---------------------------------------------------------------------------
# Resume: a unit counts as complete only at its full row count.
# ---------------------------------------------------------------------------
completed_units = set()
all_results = []

for p in [OUTPUT_FILE, PARTIAL_FILE]:
    if os.path.exists(p):
        try:
            old_df = pd.read_csv(p)
            counts = old_df.groupby(['category']).size()
            valid = [k for k, n in counts.items() if n == ROWS_PER_UNIT]
            for k in valid:
                completed_units.add(k if isinstance(k, tuple) else (k,))
            for k, n in counts.items():
                if n != ROWS_PER_UNIT:
                    print(f"Discarding partial unit {k} with {n}/{ROWS_PER_UNIT} rows.")
            keep = old_df[old_df.apply(
                lambda r: tuple(r[c] for c in ['category']) in set(
                    k if isinstance(k, tuple) else (k,) for k in valid), axis=1)]
            all_results = keep.to_dict("records")
            print(f"Resumed {len(completed_units)} completed units from {p}.")
            break
        except Exception as e:
            print(f"Could not load existing results: {e}")

# ---------------------------------------------------------------------------
# Experiment B: severity holdout
#
# Train on mild + moderate only (all five corruption types), test on severe.
# The clean-trained comparison rows for severe already exist in
# benchmark_master_combined.csv.
# ---------------------------------------------------------------------------
TRAIN_SEVS = ["mild", "moderate"]
TEST_SEV = "severe"

print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Categories: {CATEGORIES}\nTrain severities: {TRAIN_SEVS} -> test: {TEST_SEV}")

for category in CATEGORIES:
    check_timeout()
    if (category,) in completed_units:
        print(f"Skipping {category}")
        continue

    print(f"\n{'='*60}\n{category.upper()} | train {TRAIN_SEVS} -> test {TEST_SEV}\n{'='*60}")
    dm_test = MVTecAD(root=MVTEC_ROOT, category=category,
                      train_batch_size=32, eval_batch_size=32)
    dm_test.setup(stage="test")
    base_test_data = dm_test.test_data

    dst_root = os.path.join(AUG_ROOT_BASE, "mild_moderate")
    try:
        prepare_augmented_category(category, dst_root, ALL_CTYPES, TRAIN_SEVS,
                                   aug_prob=0.5, rng_seed=SEED)
        for model_name in MODELS:
            L.seed_everything(SEED)
            engine = make_engine()
            model = make_model(model_name)
            datamodule = MVTecAD(root=dst_root, category=category,
                                 train_batch_size=32, eval_batch_size=32)
            print(f"[TRAIN] {model_name} on {TRAIN_SEVS} only")
            engine.fit(model=model, datamodule=datamodule)

            for ctype in ALL_CTYPES:
                auroc = evaluate(engine, model, base_test_data, ctype, TEST_SEV)
                all_results.append({
                    "model": model_name, "dataset": "MVTec-AD", "category": category,
                    "seed": SEED, "phase": "degradation", "ctype": ctype,
                    "severity": TEST_SEV, "rescue": "none", "image_AUROC": auroc,
                    "training": "augmented_sevholdout",
                    "experiment": "severity_holdout", "held_out": TEST_SEV})
                save_results()
            del model, engine
            cleanup_memory()
        completed_units.add((category,))
    except Exception as e:
        print(f"Error in {category}: {e}")
        save_results()
        raise
    finally:
        shutil.rmtree(os.path.join(dst_root, category), ignore_errors=True)

save_results()
print(f"\n{'='*60}\nSEVERITY HOLDOUT COMPLETE - {len(all_results)} rows\n{'='*60}")


Script started at Mon Sep  7 10:36:39 2026
Graceful timeout set to 11.5 hours.
Installing missing dependency: anomalib...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.2/800.2 kB 38.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.3 which is incompatible.


Installing missing dependency: scikit-image...


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention



GPU: Tesla T4
Categories: ['carpet', 'grid', 'leather', 'bottle', 'cable', 'capsule', 'hazelnut', 'screw']
Train severities: ['mild', 'moderate'] -> test: severe

CARPET | train ['mild', 'moderate'] -> test severe


...

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │            0.5            │
│       image_F1Score       │    0.8530465960502625     │
│        pixel_AUROC        │    0.6165633201599121     │
│       pixel_F1Score       │   0.0046658883802592754   │
└───────────────────────────┴───────────────────────────┘

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.47386759519577026    │
│       image_F1Score       │    0.8530465960502625     │
│        pixel_AUROC        │    0.7003580331802368     │
│       pixel_F1Score       │   0.0073920427821576595   │
└───────────────────────────┴───────────────────────────┘


SEVERITY HOLDOUT COMPLETE - 80 rows
